# Setup Paths

In [1]:
import os

""" Inputs """
n_bits = 24 #CHANGE
mu = 50 #CHANGE
ccf_x_min = 4.5 #CHANGE
ccf_x_max = 9.5 #CHANGE

design_name = f"TreeDPNMF_{n_bits}bits" #CHANGE
design_data_path = f"/scratchdata1/ExternalData/Allen_Cortex_Hippocampus_SmartSeq_2023Sep07/adata.h5ad"
reference_data_path = f"/scratchdata1/ExternalData/Allen_WMB_2024Mar06"
simulation_data_path = [f"/scratchdata1/ExternalData/Zhaung_WMB/WB_imputation_animal1_coronal_{i}.h5ad" for i in ['anterior','posterior']]
save_path = f"/scratchdata1/GeneralStorage/Zach/Designs/{design_name}"
if not os.path.exists(save_path):
    os.mkdir(save_path)
design_path = os.path.join(save_path, 'Design')
if not os.path.exists(design_path):
    os.mkdir(design_path)
reference_path = os.path.join(save_path, 'Reference')
if not os.path.exists(reference_path):
    os.mkdir(reference_path)
simulation_path = os.path.join(save_path, 'Simulation')
if not os.path.exists(simulation_path):
    os.mkdir(simulation_path)
result_path = os.path.join(save_path, 'Results')
if not os.path.exists(result_path):
    os.mkdir(result_path)



# Design Encoding

In [ ]:
import torch
import anndata
import numpy as np
import pandas as pd
import logging
import os
from sklearn.decomposition import PCA

def get_lowdim_covmat_tree_distance(levels, meta, Xt, device='cpu'):
    """Xt is a cell by gene matrix here
    """
    # get a tree
    # L5 - L3 - L1 - L0
    # levels = [
    #    'cluster_label',
    #    'subclass_label',
    #    'class_label',
    #    ]
    if not isinstance(meta, pd.DataFrame):
        meta = pd.DataFrame(meta)
    if isinstance(Xt, np.ndarray):
        Xt = torch.from_numpy(Xt)
    Xt = Xt.to(device)
    tree = meta.groupby(levels).size()
    tree = tree[tree != 0]
    tree = tree.reset_index()[levels]
    ctrds_lvl = []
    types_lvl = []
    def group_mean(data, labels):
        unique_labels, inverse_indices = torch.unique(labels, return_inverse=True)
        num_groups = len(unique_labels)
        sums = torch.zeros((num_groups, data.shape[1]), dtype=data.dtype, device=data.device)
        counts = torch.zeros(num_groups, dtype=torch.int, device=data.device)
        for i in range(num_groups):
          mask = inverse_indices == i
          sums[i] = data[mask].sum(dim=0)
          counts[i] = mask.sum()
        means = sums / counts.unsqueeze(1)
        return means, unique_labels
    for level in levels:
        logging.info(level)
        level_data_tensor = torch.tensor(meta[level].factorize()[0], device=device)
        ctrds_, types_ = group_mean(Xt, level_data_tensor)
        logging.info(f"{ctrds_.shape}")
        logging.info(f"{types_.shape}")
        ctrds_lvl.append(ctrds_)
        types_lvl.append(types_)
    ctrds_, types_ = group_mean(Xt, torch.zeros(len(meta), dtype=torch.int64, device=device))  # All in one group
    ctrds_lvl.append(ctrds_)
    types_lvl.append(types_)
    ngenes = ctrds_lvl[0].shape[1]
    Sb = torch.zeros((ngenes, ngenes), device=device)
    l2maxmean = 0 
    for i in range(len(levels)):
        a = ctrds_lvl[i + 1]
        b = ctrds_lvl[i]
        _types = types_lvl[i]
        _lc = levels[i]
        if i + 1 < len(levels):
            _lu = levels[i + 1]
            types_map1up_df = meta.groupby(_lc)[_lu].first()
            if _types.device.type == 'cuda':
              _types_cpu = _types.cpu().numpy()
            else:
              _types_cpu = _types.numpy()
            types_map1up_df = types_map1up_df.reindex(pd.Series(_types_cpu, name=_lc))
            types_map1up = torch.tensor(types_map1up_df.values, device=device)
            types_map1up_code, _types_u = torch.unique(types_map1up, sorted=True, return_inverse=True)
            if not isinstance(types_lvl[i+1], torch.Tensor):
                types_lvl[i+1] = torch.tensor(types_lvl[i+1], device=device)
            assert torch.all(_types_u == types_lvl[i + 1].cpu())
            ctrds_diff = b - a[types_map1up_code]
        else:
            ctrds_diff = b - a.repeat(len(b), 1)
        logging.info(f'{i}, {ctrds_diff.shape}')
        l2 = torch.linalg.norm(ctrds_diff, dim=1)
        l2maxmean = max(torch.mean(l2).item(), l2maxmean)
        ctrds_diff_norm = ctrds_diff / torch.clip(l2.reshape(-1, 1), 1e-5, None)
        w_lvl = [1,1,1,1]
        Sb = Sb + w_lvl[i] * ctrds_diff_norm.T.matmul(ctrds_diff_norm)
    Sb = (l2maxmean ** 2) * Sb
    return Sb.cpu().numpy()

def initialize(X, k, init='normal', device='cpu'):
    """
    Args:
        - X: a p by n non-negative matrix (2d torch tensor)
            Note that it is the transpose of (n,p)
        - k: number of dimensions
    Output:
        - w: the weight matrix (p, k)
    """
    m, n = X.shape
    if init == 'pca':
        pca = PCA(n_components=k)
        pca.fit(X.cpu().numpy().T)
        vt = torch.from_numpy(pca.components_).to(device)
        w = torch.abs(vt.T)
    elif init == 'pca_2x':
        kh_p = int((k+1)/2)
        pca = PCA(n_components=kh_p)
        pca.fit(X.cpu().numpy().T)
        vt = torch.from_numpy(pca.components_).to(device)
        wp = torch.clip( vt.T, 0, None)
        wn = torch.clip(-vt.T, 0, None)
        w = torch.hstack([wp, wn])
        w = w[:,
            torch.vstack([torch.arange(kh_p), kh_p+torch.arange(kh_p)]).T.reshape(-1,)]
        w = w[:,:k]
    elif init == 'normal':
        w = torch.abs(torch.randn(m, k, device=device))
    elif init == 'uniform':
        w = torch.rand(m, k, device=device)
    else:
        raise ValueError("not implemented init option, chooose from: pca, pca_2x, normal, uniform")
    w = w/torch.linalg.norm(w, ord=2)
    return w

def get_PNMF(X, k,init='pca',random_seed=0, tol=1e-5, max_iter=1000,zero_tol=1e-10, verbose=False,report_stride=1,report_target_error=False,device='cpu'):
    """
    Args:
        - X: a p by n non-negative matrix (2d torch tensor)
            Note that it is the transpose of (n,p)
        - k: number of dimensions
    Output:
        - w: the weight matrix (p, k) with ||w||_2 = 1
        - record: recorded the error function every xxx time (m,2)

    ===
    optimize 1/2||X-WW^tX||_F^2
    update with
        w = w*ratio
        w = w/||w||_2
        where ratio = (2 XXt W)/(WWt XXt W + XXt WWt W)

    """
    if device == 'cuda' and torch.cuda.is_available():
      torch.cuda.manual_seed(random_seed)
      X = X.cuda()
    else:
      torch.manual_seed(random_seed)
    assert torch.any(X >= 0)
    m, n = X.shape
    k = int(k)
    assert k <= min(m, n)
    w = initialize(X, k, init=init, device=device)
    w = w/torch.linalg.norm(w, ord=2)
    xxt = X.matmul(X.T)
    record = []
    error = 1
    i = 0
    while error > tol and i < max_iter:
        wlast = w
        a = xxt.matmul(w)
        wwt = w.matmul(w.T)
        wtw = w.T.matmul(w)
        denom = wwt.matmul(a)+a.matmul(wtw)
        denom = torch.clip(denom, zero_tol, None)
        ratio = 2*a/denom
        w = w*ratio
        w = w/torch.linalg.norm(w, ord=2) # 2-norm (largest singular value) (very useful in practice)
        error = torch.linalg.norm(w-wlast, 'fro')**2
        if i % report_stride == 0:
            if verbose:
                logging.info(f"{i}, {error:.2e}")
            if report_target_error:
                target_error = torch.linalg.norm(X-w.matmul(w.T.matmul(X)), 'fro')**2
                record.append((i, error.item(), target_error.item()))
            else:
                record.append((i, error.item(),))
        i += 1
    return w, torch.tensor(record)

def get_DPNMF(X, k, s, mu,init='pca',random_seed=0, tol=1e-5, max_iter=1000,zero_tol=1e-10, verbose=False,report_stride=1,report_target_error=False,device='cpu'):
    """
    Args:
        - X: a p by n non-negative matrix (2d torch tensor)
            Note that it is the transpose of (n,p)
        - k: number of dimensions
        - s: a p by p matrix
        - mu: strength of the D term (S)
    Output:
        - w: the weight matrix (p, k) with ||w||_2 = 1
        - record: recorded the error function every xxx time (m,2)

    ===
    optimize 1/2||X-WW^tX||_F^2 + 1/2tr(W^TSW^T) with a user provided S matrix
    update with
        w = w*ratio
        w = w/||w||_2
        where ratio = (2 XXt W + Dterm)/(WWt XXt W + XXt WWt W + Dterm)

    """
    if device == 'cuda' and torch.cuda.is_available():
      torch.cuda.manual_seed(random_seed)
      X = X.cuda()
      s = s.cuda()
    else:
      torch.manual_seed(random_seed)
    assert torch.any(X >= 0)
    m, n = X.shape
    k = int(k)
    assert k <= min(m, n)
    assert (m, m) == s.shape
    w = initialize(X, k, init=init, device=device)
    sp = torch.clip( s, 0, None)
    sn = torch.clip(-s, 0, None)
    w = w/torch.linalg.norm(w, ord=2)
    xxt = X.matmul(X.T)
    record = []
    error = 1
    i = 0
    while error > tol and i < max_iter:
        wlast = w
        a = xxt.matmul(w)
        num = 2*a + mu*sn.matmul(w)
        wwt = w.matmul(w.T)
        wtw = w.T.matmul(w)
        denom = wwt.matmul(a) + a.matmul(wtw) + mu*sp.matmul(w)
        denom = torch.clip(denom, zero_tol, None)
        ratio = num/denom
        w = w*ratio
        w = w/torch.linalg.norm(w, ord=2)
        error = torch.linalg.norm(w-wlast, 'fro')**2
        if i % report_stride == 0:
            if verbose:
                logging.info(f"{i}, {error:.2e}")
                print(f"{i}, {error:.2e}")
            if report_target_error:
                target_error = torch.linalg.norm(X-w.matmul(w.T.matmul(X)), 'fro')**2
                record.append((i, error.item(), target_error.item()))
            else:
                record.append((i, error.item(),))
        i += 1
    return w, torch.tensor(record)

In [4]:
if not os.path.exists(os.path.join(design_path, f"{design_name}_design.csv")):
    """ Load Smartseq data """
    full_adata = anndata.read_h5ad(design_data_path)

    """ Class Balance """
    n = 1000
    cts = full_adata.obs['subclass_label'].unique()
    idxs = []
    for ct in cts:
        m = full_adata.obs['subclass_label'] == ct
        if m.sum() > n:
            idxs.extend(np.random.choice(np.where(m)[0], n, replace=False))
        else:
            idxs.extend(np.random.choice(np.where(m)[0], n, replace=True))
    full_adata = full_adata[idxs, :]

    """ Normalize """
    X = torch.tensor(full_adata.X)
    size = X.sum(1)
    correction = 100000/size
    X = X*correction[:,None]

    """ Filter Genes """
    unique_cell_types = full_adata.obs['subclass_label'].unique()
    gene_avg = pd.DataFrame(index=full_adata.var.index, columns=unique_cell_types)
    for cell_type in unique_cell_types:
        m = torch.tensor(full_adata.obs['subclass_label'] == cell_type)
        v = X[m].mean(0)
        gene_avg[cell_type] = v.numpy()
    gene_m = (gene_avg.max(1)>1) & (gene_avg.max(1)<100)
    X = X[:,torch.tensor(gene_m)].T

    """ Design Encoding """
    meta = full_adata.obs
    levels = ['subclass_label']
    w_lvl = [1]
    w_lvl = w_lvl/np.mean(w_lvl)
    S = torch.tensor(-1*get_lowdim_covmat_tree_distance(levels, meta, X.numpy().T))
    w, rec = get_DPNMF(X, n_bits, S, mu, init='normal', verbose=True, report_stride=1,tol=1e-6,max_iter=20)

    """ Save the result """
    WeightMat = pd.DataFrame(w,index=full_adata.var.index[gene_m])
    WeightMat.to_csv(os.path.join(design_path, f"{design_name}_design.csv"))

    """ Free up memory """
    del full_adata, X, meta, S


# Build Reference

In [5]:
import requests
import json
import os
import pathlib
import subprocess
import time
import anndata
import torch 
import pandas as pd
import numpy as np
import concurrent.futures
import threading 
from tqdm import tqdm

In [ ]:
def process_single_file(file_name, current_dataset_path, current_cell_annotation_index, current_cell_extended, current_WeightMat, current_design_name, current_projected_path, current_dataset_batch, current_dataset_name_for_path):
    """
    Processes a single data file.
    Loads data, filters cells and genes, performs projection, saves individual output, and returns AnnData object.
    """
    try:
        individual_file_out_dir = os.path.join(current_projected_path, current_dataset_batch, current_dataset_name_for_path)
        os.makedirs(individual_file_out_dir, exist_ok=True)
        out_path = os.path.join(individual_file_out_dir, file_name)
        if os.path.exists(out_path):
            print(f"File {file_name} already processed. Skipping.")
            try:
                out_data = anndata.read_h5ad(out_path)
                return out_data
            except Exception as e:
                print(f"Error reading existing output file {out_path}: {e}")
        # Optional: print(f"Processing file: {file_name} in thread {threading.get_ident()}")
        data_path = os.path.join(current_dataset_path, file_name)
        print(f"Starting processing for: {file_name}")

        # """ Load Data """
        data = anndata.read_h5ad(data_path)

        # """ Remove Cells not in annotation """
        mask_cells = data.obs.index.isin(current_cell_annotation_index)
        print(f"File {file_name}: {100*np.sum(mask_cells)/mask_cells.shape[0]:.2f}% of cells match annotation.")
        if np.sum(mask_cells) == 0:
            print(f"File {file_name}: No cells match annotation. Skipping.")
            return None
        data = data[mask_cells].copy()

        # """ Add useful info to obs """
        data.obs = current_cell_extended.loc[data.obs.index].copy() # Ensure it's a copy
        data.obs['library_size'] = data.X.sum(axis=1) # sum along axis 1 for rows

        # """ Match up genes with weights """
        # Create converter from gene symbol to gene ID for the current data object
        converter = {data.var.loc[gid]['gene_symbol']:gid for gid in data.var.index if 'gene_symbol' in data.var.columns}

        # Filter WeightMat (indexed by gene symbols) to those symbols present in the current data's gene symbols
        mask_genes_in_data = current_WeightMat.index.isin(data.var['gene_symbol'])
        filtered_WeightMat_local = current_WeightMat[mask_genes_in_data].copy()

        # Convert the gene symbols in filtered_WeightMat_local.index to GIDs using the converter
        # Only include symbols that are actually in the converter (i.e., in the current data's gene_symbols)
        valid_symbols_for_gid_conversion = [sym for sym in filtered_WeightMat_local.index if sym in converter]
        if not valid_symbols_for_gid_conversion:
            print(f"File {file_name}: No gene symbols from WeightMat found in this file's converter. Skipping.")
            return None
        
        filtered_WeightMat_local = filtered_WeightMat_local.loc[valid_symbols_for_gid_conversion]
        new_gid_index = [converter[symbol] for symbol in filtered_WeightMat_local.index]
        filtered_WeightMat_local.index = new_gid_index # Now filtered_WeightMat_local is indexed by GIDs

        print(f"File {file_name}: {100*len(new_gid_index)/len(current_WeightMat.index):.2f}% of initial WeightMat genes selected after matching with data.")

        # """ filter data """
        # Filter data to include only genes (GIDs) that are in the index of (GID-indexed) filtered_WeightMat_local
        common_gids = [gid for gid in data.var.index if gid in filtered_WeightMat_local.index]
        if not common_gids:
            print(f"File {file_name}: No common GIDs found between data and WeightMat after conversion. Skipping.")
            return None
        filtered_data = data[:, common_gids].copy()

        # """ match order """
        # Order filtered_WeightMat_local (indexed by GIDs) according to the GID order in filtered_data.var.index
        ordered_filtered_WeightMat = filtered_WeightMat_local.loc[filtered_data.var.index].copy()

        # """ project """
        projected_X = filtered_data.X.dot(ordered_filtered_WeightMat.values) # Use .values for dot product with sparse matrix

        out_data = anndata.AnnData(
            X=projected_X.astype('float32'),
            var=pd.DataFrame(ordered_filtered_WeightMat.columns, index=np.array([f"readout{i}" for i in range(ordered_filtered_WeightMat.shape[1])]), columns=['bit']),
            obs=filtered_data.obs.copy() # Use obs from filtered_data
        )
        out_data.obs['probe_set'] = current_design_name
        
        # Ensure path for individual file output exists
        individual_file_out_dir = os.path.join(current_projected_path, current_dataset_batch, current_dataset_name_for_path)
        os.makedirs(individual_file_out_dir, exist_ok=True)
        out_path = os.path.join(individual_file_out_dir, file_name)
        
        print(f"File {file_name}: Writing output to: {out_path}")
        out_data.write(out_path)
        print(f"Finished processing {file_name}")
        return out_data

    except Exception as e:
        print(f"Error processing file {file_name}: {e}")
        import traceback
        traceback.print_exc()
        return None

# Main processing block
if not os.path.exists(os.path.join(reference_path, f"{design_name}.h5ad")):
    print("Starting main processing: Combined reference file does not exist.")
    # """ Load Encoding Design"""
    print("Loading encoding design...")
    WeightMat = pd.read_csv(os.path.join(design_path, f"{design_name}_design.csv"), index_col=0)
    WeightMat = WeightMat * 9e4 / WeightMat.sum().sum()  # Scale to 90k
    print(f"Encoding design loaded. Shape: {WeightMat.shape}")

    # """ Load 10X Data Manifest and Metadata """
    print("Loading 10X data manifest and metadata...")
    version = '20230830'
    url = 'https://allen-brain-cell-atlas.s3-us-west-2.amazonaws.com/releases/%s/manifest.json' % version
    manifest = json.loads(requests.get(url).text)
    download_base = reference_data_path
    
    metadata = manifest['file_listing']['WMB-10X']['metadata']
    rpath_cell_meta = metadata['cell_metadata']['files']['csv']['relative_path']
    file_cell_meta = os.path.join(download_base, rpath_cell_meta)
    cell = pd.read_csv(file_cell_meta, dtype={'cell_label':str})
    cell.set_index('cell_label', inplace=True)

    taxonomy_metadata = manifest['file_listing']['WMB-taxonomy']['metadata']
    rpath_cluster_details = taxonomy_metadata['cluster_to_cluster_annotation_membership_pivoted']['files']['csv']['relative_path']
    file_cluster_details = os.path.join(download_base, rpath_cluster_details)
    cluster_details = pd.read_csv(file_cluster_details, keep_default_na=False)
    cluster_details.set_index('cluster_alias', inplace=True)

    rpath_cluster_colors = taxonomy_metadata['cluster_to_cluster_annotation_membership_color']['files']['csv']['relative_path']
    file_cluster_colors = os.path.join(download_base, rpath_cluster_colors)
    cluster_colors = pd.read_csv(file_cluster_colors)
    cluster_colors.set_index('cluster_alias', inplace=True)

    rpath_roi_meta = metadata['region_of_interest_metadata']['files']['csv']['relative_path']
    file_roi_meta = os.path.join(download_base, rpath_roi_meta)
    roi = pd.read_csv(file_roi_meta)
    roi.set_index('acronym', inplace=True)
    roi.rename(columns={'order':'region_of_interest_order', 'color_hex_triplet':'region_of_interest_color'}, inplace=True)

    cell_extended = cell.join(cluster_details, on='cluster_alias')
    cell_extended = cell_extended.join(cluster_colors, on='cluster_alias')
    cell_extended = cell_extended.join(roi[['region_of_interest_order', 'region_of_interest_color']], on='region_of_interest_acronym')
    print("Metadata loaded and joined.")

    data_keys = [i for i in manifest['directory_listing'].keys() if ('-10Xv' in i) and ('expression_matrices' in manifest['directory_listing'][i]['directories'].keys())]
    cell_annotation_index = cell_extended.index
    projected_path = reference_path # This is where projected data will be saved
    
    all_concatenated_data_batches = [] # To store AnnData objects from each processed batch

    for key in data_keys:
        print(f"\nProcessing key: {key}")
        data_type, dataset_batch, dataset_name_for_path = manifest['directory_listing'][key]['directories']['expression_matrices']['relative_path'].split('/')
        
        # Ensure directories for this batch exist
        os.makedirs(os.path.join(projected_path, dataset_batch, dataset_name_for_path), exist_ok=True)
        
        dataset_path_for_key = os.path.join(download_base, data_type, dataset_batch, dataset_name_for_path)
        print(f"Dataset path for key {key}: {dataset_path_for_key}")
        
        if not os.path.isdir(dataset_path_for_key):
            print(f"Warning: Dataset path {dataset_path_for_key} does not exist or is not a directory. Skipping.")
            continue
            
        files_in_dataset = os.listdir(dataset_path_for_key)

        tasks_for_executor = []
        for file_name_loop in files_in_dataset:
            if 'log' in file_name_loop.lower(): # Make check case-insensitive
                continue
            if not 'WMB' in file_name_loop: # Assuming 'WMB' check is still relevant
                continue
            if not file_name_loop.endswith('.h5ad'): # Process only .h5ad files
                print(f"Skipping non-h5ad file: {file_name_loop}")
                continue

            tasks_for_executor.append(
                (file_name_loop, dataset_path_for_key, cell_annotation_index, cell_extended, WeightMat, design_name, projected_path, dataset_batch, dataset_name_for_path)
            )
        
        if not tasks_for_executor:
            print(f"No valid .h5ad files found to process for key {key} in {dataset_path_for_key}")
            continue

        current_batch_processed_data = []
        with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
            print(f"Submitting {len(tasks_for_executor)} files for processing using 5 threads for batch {dataset_batch}...")
            future_to_task_args = {executor.submit(process_single_file, *task_args): task_args for task_args in tasks_for_executor}

            for i, future in tqdm(enumerate(concurrent.futures.as_completed(future_to_task_args)), total=len(future_to_task_args), desc=f"Processing files in batch {dataset_batch}"):
                task_args_done = future_to_task_args[future]
                file_name_done = task_args_done[0]
                print(f"Thread finished for file: {file_name_done} ({i+1}/{len(tasks_for_executor)})")
                try:
                    result_out_data = future.result()
                    if result_out_data is not None:
                        current_batch_processed_data.append(result_out_data)
                except Exception as exc:
                    print(f"File {file_name_done} generated an exception during future.result(): {exc}")
                    import traceback
                    traceback.print_exc()
        
        if current_batch_processed_data:
            print(f"Concatenating {len(current_batch_processed_data)} processed files for batch {dataset_batch}...")
            try:
                concatenated_data_for_batch = anndata.concat(current_batch_processed_data, index_unique='observations') # Or 'raise' if preferred
                
                # Ensure batch output directory exists
                batch_out_dir = os.path.join(projected_path, dataset_batch)
                os.makedirs(batch_out_dir, exist_ok=True)
                out_path_batch_combined = os.path.join(batch_out_dir, dataset_batch + '_combined.h5ad')
                
                print(f"Writing combined batch data to: {out_path_batch_combined}")
                concatenated_data_for_batch.write(out_path_batch_combined)
                all_concatenated_data_batches.append(concatenated_data_for_batch)
                print(f"Finished processing and combining for batch {dataset_batch}")
            except Exception as e_concat:
                print(f"Error concatenating batch {dataset_batch}: {e_concat}")
                import traceback
                traceback.print_exc()

        else:
            print(f"No data to concatenate for batch {dataset_batch}.")
        print(' ') # Original spacing

    if all_concatenated_data_batches:
        print("\nConcatenating all processed batches...")
        try:
            final_all_concatenated_data = anndata.concat(all_concatenated_data_batches, index_unique='observations') # Or 'raise'
            final_output_path = os.path.join(projected_path, f"{design_name}.h5ad")
            print(f"Writing final concatenated data to: {final_output_path}")
            final_all_concatenated_data.write(final_output_path)
            print("All processing complete. Final file written.")
            del final_all_concatenated_data # Free memory
        except Exception as e_final_concat:
            print(f"Error during final concatenation: {e_final_concat}")
            import traceback
            traceback.print_exc()
    else:
        print("No data was processed and concatenated across all batches. Final file not written.")

    # """ Free up memory """
    print("Freeing up memory...")
    if 'all_concatenated_data_batches' in locals(): del all_concatenated_data_batches
    if 'cell_extended' in locals(): del cell_extended
    if 'cell' in locals(): del cell
    if 'cluster_details' in locals(): del cluster_details
    if 'cluster_colors' in locals(): del cluster_colors
    if 'roi' in locals(): del roi
    if 'manifest' in locals(): del manifest
    if 'WeightMat' in locals(): del WeightMat
    print("Memory cleanup attempted.")
else:
    fname = os.path.join(reference_path, f"{design_name}.h5ad")
    print(f"Combined reference file {fname} already exists. Skipping processing.")

# Build Simulation

In [7]:
import anndata
import numpy as np
import pandas as pd
import torch

In [ ]:
""" PreFormat simulation_path """
""" Load Zhuang Data"""
from scipy.sparse import csr_matrix
from tqdm import trange
projected_adatas = []
if isinstance(simulation_data_path, str):
    simulation_data_path = [simulation_data_path]
for fname in simulation_data_path:
    adata = anndata.read_h5ad(fname,backed='r') #CHANGE
    # break into 50k cell chunks
    n_cells = adata.shape[0]
    n_chunks = int(np.ceil(n_cells / 50000))
    # var = adata.var.copy()

    for i in trange(n_chunks):
        if os.path.exists(fname.replace('.h5ad', f'_chunk{i}.h5ad')):
            try:
                temp_adata = anndata.read_h5ad(fname.replace('.h5ad', f'_chunk{i}.h5ad'),backed='r')
                print(f"Chunk {i} already exists, skipping.")
                continue
            except:
                print(f"Chunk {i} does not exist, creating it.")
        # adata = anndata.read_h5ad(fname,backed='r')
        # n_cells = adata.shape[0]
        # var = adata.var.copy()
        start_idx = i * 50000
        end_idx = min((i + 1) * 50000, n_cells)
        # obs = adata.obs.iloc[start_idx:end_idx].copy()
        # X = adata.X[start_idx:end_idx].copy()  # Use .copy() to ensure it's a dense matrix
        chunk_adata = anndata.AnnData(
            X=adata.X[start_idx:end_idx],  # Ensure X is a sparse matrix
            obs=adata.obs.iloc[start_idx:end_idx],
            var=adata.var,
        )
        chunk_adata.obsm['X_CCF'] = adata.obsm['X_CCF'][start_idx:end_idx]
        chunk_adata.write(fname.replace('.h5ad', f'_chunk{i}.h5ad'))


In [ ]:
import os
import pandas as pd
import numpy as np
import anndata
import torch
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


def process_chunk_item(chunk_h5ad_path, WeightMat_main, ccf_x_min_val, ccf_x_max_val):
    """Processes a single data chunk file."""
    try:
        adata = anndata.read_h5ad(chunk_h5ad_path)
        
        ccf_x_coords = np.array(adata.obsm['X_CCF'])[:, 0] / 1000.0
        mask = (adata.obs['high_quality_transfer']) & (ccf_x_coords > ccf_x_min_val) & (ccf_x_coords < ccf_x_max_val)
        
        # Ensure there are cells left after filtering
        if not np.any(mask):
            # print(f"No cells left in {os.path.basename(chunk_h5ad_path)} after CCF/quality filtering.")
            return None
        
        adata = adata[mask, :].copy()

        adata_genes = adata.var['gene_name']
        shared_genes = sorted(list(set(adata_genes).intersection(set(WeightMat_main.index))))

        if not shared_genes:
            # print(f"No shared genes for {os.path.basename(chunk_h5ad_path)}")
            return None

        # Prepare var for gene ID mapping
        var_temp = adata.var.copy()
        var_temp['gene_id_col'] = var_temp.index # Store original var index (e.g. Ensembl)
        var_temp = var_temp.set_index('gene_name', drop=False) # Set gene_name as index
        var_temp_shared = var_temp.loc[shared_genes]
        adata_var_indices_ordered = var_temp_shared['gene_id_col'].values
        
        adata = adata[:, adata_var_indices_ordered].copy()
        
        # Filter WeightMat for the shared genes in this chunk
        WeightMat_chunk_specific = WeightMat_main.loc[shared_genes]

        E = torch.tensor(WeightMat_chunk_specific.values, dtype=torch.float32)
        
        # Ensure X is a dense array for tensor conversion
        if hasattr(adata.X, "toarray"):
            X_data = adata.X.toarray()
        else:
            X_data = adata.X
        X = torch.tensor(X_data, dtype=torch.float32)
        
        # Metadata for projected AnnData
        y_str = np.array(adata.obs['subclass_transfer'])
        ccf_coords = adata.obsm['X_CCF']
        
        # Project
        P = X.mm(E)
        
        projected_obs_df = pd.DataFrame(index=adata.obs.index)
        projected_obs_df['subclass'] = y_str
        projected_obs_df['ccf_x'] = ccf_coords[:, 0] / 1000.0
        projected_obs_df['ccf_y'] = ccf_coords[:, 1] / 1000.0
        projected_obs_df['ccf_z'] = ccf_coords[:, 2] / 1000.0
        
        num_readouts = WeightMat_chunk_specific.shape[1]
        projected_var_df = pd.DataFrame(index=[f"readout{i}" for i in range(num_readouts)])
        projected_var_df['readout'] = [f"readout{i}" for i in range(num_readouts)]
        projected_var_df['hybe'] = [f"hybe{i}" for i in range(num_readouts)]
        projected_var_df['channel'] = [f"FarRed" for i in range(num_readouts)]
        
        return anndata.AnnData(X=P.numpy(), obs=projected_obs_df.copy(), var=projected_var_df.copy())
    except Exception as e:
        print(f"Error processing {os.path.basename(chunk_h5ad_path)}: {e}")
        return None

# Main processing logic
output_file = os.path.join(simulation_path, f"{design_name}.h5ad")

if not os.path.exists(output_file):
    # Load Encoding Design
    WeightMat = pd.read_csv(os.path.join(design_path, f"{design_name}_design.csv"), index_col=0)
    WeightMat = WeightMat * 9e4 / WeightMat.sum().sum()  # Scale to 90k
    
    all_projected_adatas = []
    
    if isinstance(simulation_data_path, str):
        simulation_data_path_list = [simulation_data_path]
    else:
        simulation_data_path_list = simulation_data_path

    chunk_file_paths_to_process = []
    for fname_base in simulation_data_path_list:
        try:
            # Read base file just to get number of cells and thus chunks
            adata_main_ref = anndata.read_h5ad(fname_base, backed='r')
            n_cells = adata_main_ref.shape[0]
            del adata_main_ref # free memory
            n_chunks = int(np.ceil(n_cells / 50000))
            for i in range(n_chunks):
                chunk_file = fname_base.replace('.h5ad', f'_chunk{i}.h5ad')
                if os.path.exists(chunk_file): # Ensure chunk file exists
                    chunk_file_paths_to_process.append(chunk_file)
                # else:
                    # print(f"Warning: Chunk file {chunk_file} not found.")
        except Exception as e:
            print(f"Error reading base file {fname_base} to determine chunks: {e}")
            continue # Skip this base file if it can't be read

    if not chunk_file_paths_to_process:
        print("No chunk files found to process.")
    else:
        with ThreadPoolExecutor(max_workers=10) as executor:
            # Submit all tasks
            futures = [executor.submit(process_chunk_item, cfp, WeightMat, ccf_x_min, ccf_x_max) for cfp in chunk_file_paths_to_process]
            
            # Process as they complete with a progress bar
            for future in tqdm(as_completed(futures), total=len(futures), desc="Processing chunks"):
                result = future.result()
                if result is not None:
                    all_projected_adatas.append(result)
        
        if all_projected_adatas:
            final_projected_adata = anndata.concat(all_projected_adatas, axis=0, join='outer', merge='same')
            final_projected_adata.write_h5ad(output_file) # Write the final concatenated file
            print(f"Processing complete. Final AnnData shape: {final_projected_adata.shape}")
            # Clean up memory
            del final_projected_adata, all_projected_adatas, WeightMat
        else:
            print("No data was successfully processed from chunks.")
else:
    print(f"Output file {output_file} already exists. Skipping.")

# Test Encoding

In [11]:
from scipy.interpolate import interp1d
from ATLAS.Analysis.Classification import *


In [ ]:
""" Load Simulation Data """
adata = anndata.read_h5ad(os.path.join(simulation_path,f"{design_name}.h5ad"))
adata = adata[(adata.obs['ccf_x']>ccf_x_min) &(adata.obs['ccf_x']<ccf_x_max)].copy()
adata.obs['true_subclass'] = adata.obs['subclass'].copy()
adata.layers['raw'] = adata.X.copy()

""" Load Reference"""
complete_reference = anndata.read_h5ad(os.path.join(reference_path,f"{design_name}.h5ad"))

""" Decode """
np.random.seed(42)
self = SingleCellAlignmentLeveragingExpectations(adata,complete_reference=complete_reference,visualize=False,verbose=False)
self.likelihood_only = False
self.calculate_spatial_priors()
self.load_reference()
self.model = LogisticRegression(max_iter=1000,random_state=42) 
self.supervised_neuron_annotation()
self.supervised_harmonization()
adata = self.measured.copy()
remove_index = {ct:ct[4:] for ct in adata.obs['subclass'].unique()}
adata.obs['predicted_subclass'] = adata.obs['subclass'].map(remove_index)
adata.write(os.path.join(result_path,f"{design_name}.h5ad"))
accuracy = np.mean(np.array(adata.obs['predicted_subclass'].values)==np.array(adata.obs['true_subclass'].values))
print(accuracy)

In [ ]:
print(accuracy)